# Verify video pipeline

Confirm a raw LRS3-trainval frame, after the grayscale transform the
frozen appearance encoder expects (see CLAUDE.md's "Precisely where and
why grayscale happens" note -- grayscale is applied at data-LOADING time,
not baked into the saved crop files), ends up the same shape/format as
test-mattymchen's native, already-preprocessed frames.

**What "looks right" means:**
- After grayscale + resize/crop, a trainval frame's tensor shape matches
  test-mattymchen's native `(96, 96)` frame shape exactly.
- dtype and value range (e.g. `uint8` 0-255 vs `float32` 0-1) are
  consistent between the two, or the mismatch is understood and
  intentional.
- The two frames look like reasonably consistent mouth crops side by
  side -- similar framing/scale, not wildly different zoom levels or
  face positions. test-mattymchen went through an unknown external
  preprocessing pipeline, so this is a one-time eyeball check, not
  something to assume matches without looking.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import torchvision
import torchvision.transforms as T
from datasets import load_dataset

## Config

In [ ]:
# TODO: fill in a real path from notebook 01's lrs3_trainval manifest video_path column
SAMPLE_TRAINVAL_VIDEO_PATH = Path("/scratch/project_2020712/datasets/lrs3/ainncy/trainval/REPLACE_ME/REPLACE_ME.mp4")
MATTYMCHEN_PARQUET_DIR = Path("/scratch/project_2020712/datasets/lrs3/test-mattymchen/data")

## Load one raw LRS3-trainval frame

In [ ]:
video_frames, _audio, _info = torchvision.io.read_video(str(SAMPLE_TRAINVAL_VIDEO_PATH), pts_unit="sec")
raw_frame = video_frames[0]  # (H, W, C), uint8, RGB
print(f"raw frame shape: {tuple(raw_frame.shape)}, dtype: {raw_frame.dtype}")

## Apply the grayscale + resize/crop transform

TODO: this should match whatever `fusion_avsr`'s data-loading transform
pipeline eventually implements for the appearance encoder (mirroring
auto-AVSR's `transforms.py`) -- fill in the real crop/resize logic once
that exists; the `Grayscale()` + `Resize((96, 96))` below is a
placeholder standing in for it.

In [ ]:
transform = T.Compose([
    T.ToPILImage(),
    T.Grayscale(),
    T.Resize((96, 96)),  # TODO: replace with the real mouth-crop logic, not a naive resize
    T.PILToTensor(),
])

# torchvision frames are (H, W, C); ToPILImage expects (C, H, W) or (H, W, C) - confirm below
processed_frame = transform(raw_frame.permute(2, 0, 1))
print(f"processed frame shape: {tuple(processed_frame.shape)}, dtype: {processed_frame.dtype}")

## Load one test-mattymchen frame for comparison

In [ ]:
mattymchen = load_dataset("parquet", data_dir=str(MATTYMCHEN_PARQUET_DIR))
mattymchen_split = mattymchen["train"] if "train" in mattymchen else next(iter(mattymchen.values()))

mattymchen_example = mattymchen_split[0]
mattymchen_frame = mattymchen_example["video"][0]  # first frame, native (96, 96)
print(f"mattymchen frame shape: {len(mattymchen_frame)}x{len(mattymchen_frame[0])}")

## Visual side-by-side

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(processed_frame.squeeze(0), cmap="gray")
axes[0].set_title("processed trainval crop")
axes[1].imshow(mattymchen_frame, cmap="gray")
axes[1].set_title("test-mattymchen native frame")
for ax in axes:
    ax.axis("off")
plt.show()

## TODO checklist

- [ ] `processed_frame.shape[-2:] == (96, 96)`, matching test-mattymchen's
      native frame shape.
- [ ] Both frames are single-channel (grayscale).
- [ ] Crop framing/scale look reasonably consistent side by side (not
      wildly different zoom/position) -- if they diverge visibly, that's
      a provenance difference (test-mattymchen's unknown external
      pipeline), not necessarily a bug in our own pipeline.